# 采购入库数据清洗确认

## 1. 读取原始数据

In [1]:
import pandas as pd
import numpy as np
from read_data import read_raw_procurement_inbound

df = read_raw_procurement_inbound("2025-10-01", "2026-09-30")
print(f"{len(df):,} 行，{len(df.columns)} 列")
df

356,164 行，51 列


,分析类型,数据来源系统,年份,月份,公司编码,公司名称,公司简称,板块,多组织名称,大区,...,三级分类,四级分类,五级分类,自定义内外部供应商（凯莱谱、生物为外部）,设备试剂分类,财务内部外供应商类型,是否多组织内交易,采购类型名称,_source,_fetched_at
0,入库,nc,2025,10,2012,武汉迪安医学检验实验室有限公司,武汉迪安,检验,None,华中片区,...,通用耗材,无分类,无分类,外部供应商,试剂/耗材,外部单位,0,自行采购合同,procurement-inbound,2026-10-30T08:57:34+08:00
1,入库,nc,2025,10,2003,南京迪安医学检验所有限公司,南京迪安,检验,None,华东片区,...,试剂,临床化学检验专业,生化,外部供应商,试剂/耗材,外部单位,0,定向采购合同,procurement-inbound,2026-10-30T08:57:34+08:00
2,入库,nc,2025,10,2004,上海迪安医学检验所有限公司,上海迪安,检验,None,华东片区,...,试剂,临床免疫检验专业,发光,外部供应商,试剂/耗材,外部单位,0,定向采购合同,procurement-inbound,2026-10-30T08:57:34+08:00
3,入库,nc,2025,10,2001,杭州迪安医学检验中心有限公司,杭州迪安,检验,None,华东片区,...,试剂,临床免疫检验专业,发光,外部供应商,试剂/耗材,外部单位,0,定向采购合同,procurement-inbound,2026-10-30T08:57:34+08:00
4,入库,nc,2025,10,1001,迪安诊断技术集团股份有限公司,迪安诊断,诊断,None,平台,...,试剂,临床免疫检验专业,发光,外部供应商,试剂/耗材,外部单位,0,集中采购合同,procurement-inbound,2026-10-30T08:57:34+08:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356159,入库,ebs,2026,9,3076,广州迪安医学检验科技有限公司,广迪科技,检验产品,NaN,华南片区,...,试剂,临床血液与体液检验专业,流式细胞,外部供应商,试剂/耗材,外部单位,0,定向采购,procurement-inbound,2026-10-30T08:57:34+08:00
356160,入库,nc,2026,9,2015,云南迪安医学检验所有限公司,云南迪安,检验,NaN,西南片区,...,试剂,临床细胞分子遗传学专业,PCR,内部供应商,试剂/耗材,内部关联单位,0,指定采购合同,procurement-inbound,2026-10-30T08:57:34+08:00
356161,入库,nc,2026,9,1001,迪安诊断技术集团股份有限公司,迪安诊断,诊断,NaN,平台,...,试剂,临床免疫检验专业,发光,外部供应商,试剂/耗材,外部单位,0,集中采购合同,procurement-inbound,2026-10-30T08:57:34+08:00
356162,入库,ebs,2026,9,3002,杭州迪安基因工程有限公司(B),基因工程B,检验产品,NaN,平台,...,试剂,临床免疫检验专业,发光,外部供应商,试剂/耗材,外部单位,0,指定采购,procurement-inbound,2026-10-30T08:57:34+08:00


## 2. 已确认字段概览

购退单价仅用于补充采购单价。

In [2]:
字段 = [
    "出入库日期", "物料编码", "物料名称", "规格", "主单位",
    "一级分类", "二级分类", "三级分类", "四级分类", "五级分类",
    "品牌", "主品牌", "货号", "供应商名称", "财务内部外供应商类型", "板块",
    "公司编码", "公司名称", "公司简称", "大区", "数据来源系统", "设备试剂分类",
    "采购类型名称", "数量", "采购单价", "采购金额(万)", "税率", "购退单价",
]
pd.DataFrame({"类型": df[字段].dtypes.astype(str), "空值": df[字段].isna().sum(),
              "空值占比": df[字段].isna().mean(), "取值数": df[字段].nunique()})

,类型,空值,空值占比,取值数
出入库日期,str,0,0.000000,343
物料编码,int64,0,0.000000,22152
物料名称,str,0,0.000000,15943
规格,str,0,0.000000,11065
主单位,str,0,0.000000,39
一级分类,str,0,0.000000,5
二级分类,str,0,0.000000,10
三级分类,str,0,0.000000,34
四级分类,str,0,0.000000,19
五级分类,str,0,0.000000,101


## 3. 字段验证

每项筛选异常记录到 `明细`。

### 财务内部外供应商类型

空值填“外部单位”。

In [3]:
明细 = df.loc[df["财务内部外供应商类型"].isna(), 字段]
print(f"空值：{len(明细):,} 行")
明细

空值：8,118 行


,出入库日期,物料编码,物料名称,规格,主单位,一级分类,二级分类,三级分类,四级分类,五级分类,...,公司简称,大区,数据来源系统,设备试剂分类,采购类型名称,数量,采购单价,采购金额(万),税率,购退单价
8,2025-10-01,11048409,甲胎蛋白测定试剂盒 AFP Reagent Kit（）,Alinity: 2×100 测试/盒,盒,体外诊断产品,体外诊断试剂,试剂,临床免疫检验专业,发光,...,杭州迪安,华东片区,nc,试剂/耗材,定向采购合同,2.0,1144.64,0.228928,0.13,NaN
41,2025-10-01,11068506,医用检查手套（乳胶医用检查手套无粉(厚款)）,"类型1(乳胶型):M(7.5)100只/盒,10盒/箱",箱,通用耗材,通用耗材,防护用品,无分类,无分类,...,北京迪安,华北片区,nc,试剂/耗材,定向采购合同,22.0,220.00,0.484000,0.13,NaN
45,2025-10-01,11067995,全血 RNA 提取试剂盒（组份 A）,50次,盒,体外诊断产品,体外诊断试剂,试剂,临床细胞分子遗传学专业,测序,...,杭州迪安,华东片区,nc,试剂/耗材,定向采购合同,3.0,1875.00,0.562500,0.13,NaN
62,2025-10-01,11003686,细菌鉴定药敏分析试剂盒（）,"肠杆菌科药敏试剂盒,10人份/盒",盒,体外诊断产品,体外诊断试剂,试剂,临床微生物检验专业,微生物,...,江西迪安,华东片区,nc,试剂/耗材,定向采购合同,10.0,95.00,0.095000,0.13,NaN
97,2025-10-01,11023994,样本稀释液（真菌鉴定培养基）,"缓冲型,10瓶/盒,5ml/瓶",盒,体外诊断产品,体外诊断试剂,配套试剂/耗材,临床微生物检验专业,微生物,...,成都迪安,西南片区,nc,试剂/耗材,定向采购合同,2.0,0.00,0.000000,0.13,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356119,2026-09-30,11044543,游离三碘甲状腺原氨酸测定试剂盒 Free T3 Reagent Kit,ARCHITECT:1×100 测试/盒,盒,体外诊断产品,体外诊断试剂,试剂,临床免疫检验专业,发光,...,浙江医疗,平台,ebs,试剂/耗材,定向采购,3.0,483.91,0.145173,0.13,NaN
356120,2026-09-30,11003685,细菌鉴定药敏分析试剂盒（）,"非发酵菌药敏试剂盒,10人份/盒",盒,体外诊断产品,体外诊断试剂,试剂,临床微生物检验专业,微生物,...,广西医疗设备,华南片区,ebs,试剂/耗材,定向采购,1.0,95.00,0.009500,0.13,NaN
356136,2026-09-30,11044548,总甲状腺素测定试剂盒 Total T4 Reagent Kit,ARCHITECT:1×100 测试/盒,盒,体外诊断产品,体外诊断试剂,试剂,临床免疫检验专业,发光,...,浙江医疗,平台,ebs,试剂/耗材,定向采购,3.0,483.91,0.145173,0.13,NaN
356137,2026-09-30,11048402,C 肽测定试剂盒 C-Peptide Reagent Kit（）,Alinity:2×100 测试/盒,盒,体外诊断产品,体外诊断试剂,试剂,临床免疫检验专业,发光,...,浙江医疗,平台,ebs,试剂/耗材,定向采购,2.0,1272.00,0.254400,0.13,NaN


### 采购单价与购退单价

采购单价为空时用购退单价补充。

In [4]:
明细 = df.loc[df["采购单价"].isna() & df["购退单价"].notna(), 字段]
print(f"可补充：{len(明细):,} 行")
明细

可补充：998 行


,出入库日期,物料编码,物料名称,规格,主单位,一级分类,二级分类,三级分类,四级分类,五级分类,...,公司简称,大区,数据来源系统,设备试剂分类,采购类型名称,数量,采购单价,采购金额(万),税率,购退单价
3100,2025-10-09,11003494,氨苄西林药敏实验纸片 Ampicillin Susceptibility Test Disc（）,5×50片/盒,盒,体外诊断产品,体外诊断试剂,试剂,临床微生物检验专业,微生物,...,成都蓉康,西南片区,ebs,试剂/耗材,自行采购,-32.0,NaN,-0.556800,0.13,174.00
3102,2025-10-09,11003489,氨曲南药敏实验纸片 Aztreonam Susceptibility Test Disc（）,5×50片/盒,盒,体外诊断产品,体外诊断试剂,试剂,临床微生物检验专业,微生物,...,成都蓉康,西南片区,ebs,试剂/耗材,自行采购,-40.0,NaN,-0.696000,0.13,174.00
3191,2025-10-09,11003538,庆大霉素药敏实验纸片 Gentamicin Susceptibility Test Disc,5×50片/盒,盒,体外诊断产品,体外诊断试剂,试剂,临床微生物检验专业,微生物,...,成都蓉康,西南片区,ebs,试剂/耗材,自行采购,-40.0,NaN,-0.696000,0.13,174.00
3372,2025-10-09,11003479,磺胺甲恶唑/甲氧苄啶药敏实验纸片 Sulphamethoxazole/trimethopri...,50片/筒×5筒,盒,体外诊断产品,体外诊断试剂,试剂,临床微生物检验专业,微生物,...,成都蓉康,西南片区,ebs,试剂/耗材,自行采购,-16.0,NaN,-0.278400,0.13,174.00
4189,2025-10-10,11039576,促甲状腺素受体抗体检测试剂盒 Elecsys Anti-TSHR（cobas e 411/6...,100测试/盒,盒,体外诊断产品,体外诊断试剂,试剂,临床免疫检验专业,发光,...,浙江医疗,平台,ebs,试剂/耗材,集中采购,-2.0,NaN,-0.438750,0.13,2193.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
354900,2026-09-29,11044778,缓冲液（特定蛋白分析仪用(CRP)）,3.5L×1/盒,盒,体外诊断产品,体外诊断试剂,配套试剂/耗材,无分类,无分类,...,基因工程B,平台,ebs,试剂/耗材,指定采购,-10.0,NaN,-0.105000,0.13,105.00
355157,2026-09-29,11068042,尿液分析试纸条（适用于 FUS-3000Plus/MUS-3600/MUS-9600）,"FUS-12MA II:100条/筒,10筒/盒",盒,体外诊断产品,体外诊断试剂,试剂,临床血液与体液检验专业,尿常规,...,成都蓉康,西南片区,ebs,试剂/耗材,定向采购,-63.0,NaN,-5.355000,0.13,850.00
355773,2026-09-30,11062213,一次性使用拭子（塑棒）,"女性拭子,150支/袋,12袋/箱",箱,通用耗材,通用耗材,采样耗材,无分类,无分类,...,内蒙同景,东北片区,ebs,试剂/耗材,指定采购,-5.0,NaN,-0.270000,0.13,540.00
355859,2026-09-30,11022553,B 群链球菌显色平板,70MM 20块/盒,盒,体外诊断产品,体外诊断试剂,试剂,临床微生物检验专业,微生物,...,浙江医疗,平台,ebs,试剂/耗材,集中采购,-4.0,NaN,-0.069552,0.13,173.88


### 采购单价负值

方案要求修正，当前清洗脚本尚未实现。

In [5]:
明细 = df.loc[df["采购单价"].lt(0), 字段]
print(f"负单价：{len(明细):,} 行")
明细

负单价：0 行


,出入库日期,物料编码,物料名称,规格,主单位,一级分类,二级分类,三级分类,四级分类,五级分类,...,公司简称,大区,数据来源系统,设备试剂分类,采购类型名称,数量,采购单价,采购金额(万),税率,购退单价


### 采购单价与采购金额

含税单价 × 数量与含税金额的相对偏差超过 1%。

In [6]:
偏差 = (df["采购单价"] * df["数量"] / 10000 - df["采购金额(万)"]).abs() / df["采购金额(万)"].abs().replace(0, np.nan)
明细 = df.loc[偏差.gt(0.01), 字段]
print(f"偏差超过 1%：{len(明细):,} 行；无法计算：{偏差.isna().sum():,} 行")
明细

偏差超过 1%：8 行；无法计算：41,385 行


,出入库日期,物料编码,物料名称,规格,主单位,一级分类,二级分类,三级分类,四级分类,五级分类,...,公司简称,大区,数据来源系统,设备试剂分类,采购类型名称,数量,采购单价,采购金额(万),税率,购退单价
1750,2025-10-01,11017827,谷氨酸脱羧酶抗体测定试剂盒,2×50人份/盒,盒,体外诊断产品,体外诊断试剂,试剂,临床免疫检验专业,发光,...,杭州迪安,华东片区,nc,试剂/耗材,集中采购合同,70.0,618.83,8.663620,0.13,NaN
59575,2025-12-04,11035125,抗角蛋白抗体(AKA)检测试剂盒（）,50人份/盒,盒,体外诊断产品,体外诊断试剂,试剂,临床免疫检验专业,荧光,...,长沙迪安,华南片区,nc,试剂/耗材,定向采购合同,1.0,297.50,0.089250,0.13,NaN
118763,2026-02-01,11068506,医用检查手套（乳胶医用检查手套无粉(厚款)）,"类型1(乳胶型):M(7.5)100只/盒,10盒/箱",箱,通用耗材,通用耗材,防护用品,无分类,无分类,...,温州迪安,华东片区,nc,试剂/耗材,定向采购合同,7.0,220.00,0.132000,0.13,NaN
138755,2026-03-03,11003300,结核分枝杆菌核酸检测试剂盒（）,48人份/盒,盒,体外诊断产品,体外诊断试剂,试剂,临床细胞分子遗传学专业,PCR,...,南京迪安,华东片区,nc,试剂/耗材,定向采购合同,10.0,288.00,0.144000,0.13,NaN
195893,2026-04-30,11039407,一次性使用橡胶检查手套,"型号:M无粉 100支/盒,10盒/箱",箱,通用耗材,通用耗材,防护用品,无分类,无分类,...,昆山迪安,华东片区,nc,试剂/耗材,定向采购合同,0.0,740.00,0.400932,0.13,NaN
200663,2026-05-07,11027307,超氧化物歧化酶测定试剂盒,R1:1x60ml R2:1x20ml,盒,体外诊断产品,体外诊断试剂,试剂,临床化学检验专业,生化,...,杭州迪安,华东片区,nc,试剂/耗材,定向采购合同,2.0,450.00,0.135000,0.13,NaN
200923,2026-05-07,11027311,超氧化物歧化酶测定试剂盒-质控品,2x1ml,盒,体外诊断产品,体外诊断试剂,质控,临床化学检验专业,生化,...,杭州迪安,华东片区,nc,试剂/耗材,定向采购合同,3.0,15.00,0.003000,0.13,NaN
226538,2026-06-01,11006424,抗卵巢抗体检测试剂盒（）,48人份/盒,盒,体外诊断产品,体外诊断试剂,试剂,临床免疫检验专业,酶免,...,广州迪安,华南片区,nc,试剂/耗材,定向采购合同,4.0,114.66,0.034398,0.13,NaN


### 采购类型名称

空值标为“未标注”。

In [7]:
明细 = df.loc[df["采购类型名称"].isna(), 字段]
print(f"空值：{len(明细):,} 行")
明细

空值：4,152 行


,出入库日期,物料编码,物料名称,规格,主单位,一级分类,二级分类,三级分类,四级分类,五级分类,...,公司简称,大区,数据来源系统,设备试剂分类,采购类型名称,数量,采购单价,采购金额(万),税率,购退单价
3081,2025-10-09,12020089,分杯处理系统,CO-Auto Scr-Pipe-96-05,台,体外诊断产品,体外诊断设备,辅助设备,样本处理设备,样本处理系统,...,上海迪浦康,华东片区,ebs,设备/配件,NaN,1.0,30000.0,3.0000,0.13,NaN
3088,2025-10-09,12019714,全自动血细胞分析仪,F 680P,台,体外诊断产品,体外诊断设备,检测设备,血液体液检验设备,血细胞分析仪,...,沈阳医疗,东北片区,ebs,设备/配件,NaN,1.0,80000.0,8.0000,0.13,NaN
3131,2025-10-09,12020196,等离子体空气消毒机（壁挂1000智联款）,B-ZKDY-1000-Y1,台,体外诊断产品,体外诊断设备,辅助设备,通用辅助设备,消毒灭菌设备,...,合肥迪赛金,华东片区,ebs,设备/配件,NaN,1.0,2960.0,0.2960,0.13,NaN
3138,2025-10-09,12009705,全自动核酸提取仪,BG-Abot-96,台,体外诊断产品,体外诊断设备,检测设备,分子遗传检验设备,核酸提取纯化仪,...,上海迪浦康,华东片区,ebs,设备/配件,NaN,1.0,18000.0,1.8000,0.13,NaN
3237,2025-10-09,12010857,高频 UPS,ST6KVA,台,体外诊断产品,体外诊断设备,辅助设备,通用辅助设备,UPS,...,合肥迪赛金,华东片区,ebs,设备/配件,NaN,1.0,4850.0,0.4850,0.13,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355940,2026-09-30,12019415,基因测序仪,DNBSEQ-E25,台,体外诊断产品,体外诊断设备,检测设备,分子遗传检验设备,基因测序仪,...,浙江医疗,平台,ebs,设备/配件,NaN,1.0,133000.0,13.3000,0.13,NaN
356041,2026-09-30,12017762,微孔板热封仪,ASP1000,台,体外诊断产品,体外诊断设备,辅助设备,通用辅助设备,其他辅助类设备,...,云南迪安（产品）,西南片区,nc,设备/配件,NaN,1.0,3523.0,0.3523,0.13,NaN
356093,2026-09-30,12021611,实时荧光定量 PCR 分析仪,SLAN-48S,台,体外诊断产品,体外诊断设备,检测设备,分子遗传检验设备,核酸扩增仪（定量）,...,广迪科技,华南片区,ebs,设备/配件,NaN,1.0,0.0,0.0000,0.13,NaN
356129,2026-09-30,12021554,全自动核酸提取仪,S-S13H,台,体外诊断产品,体外诊断设备,检测设备,分子遗传检验设备,核酸提取纯化仪,...,广迪科技,华南片区,ebs,设备/配件,NaN,1.0,500000.0,50.0000,0.13,NaN


### 关键字段空值

物料编码、数量、金额、日期、板块或供应商名称为空。

In [8]:
关键字段 = ["物料编码", "数量", "采购金额(万)", "出入库日期", "板块", "供应商名称"]
明细 = df.loc[df[关键字段].isna().any(axis=1), 字段]
print(f"存在空值：{len(明细):,} 行")
明细

存在空值：0 行


,出入库日期,物料编码,物料名称,规格,主单位,一级分类,二级分类,三级分类,四级分类,五级分类,...,公司简称,大区,数据来源系统,设备试剂分类,采购类型名称,数量,采购单价,采购金额(万),税率,购退单价


### 文本字段格式

检查首尾空格、回车、换行和制表符。

In [9]:
文本字段 = df[字段].select_dtypes(include=["object", "string", "str"]).columns
条件 = pd.Series(False, index=df.index)
for 列 in 文本字段:
    条件 |= df[列].astype("string").str.contains(r"[\r\n\t]|^\s|\s$", regex=True, na=False)
明细 = df.loc[条件, 字段]
print(f"存在格式问题：{len(明细):,} 行")
明细

存在格式问题：458 行


,出入库日期,物料编码,物料名称,规格,主单位,一级分类,二级分类,三级分类,四级分类,五级分类,...,公司简称,大区,数据来源系统,设备试剂分类,采购类型名称,数量,采购单价,采购金额(万),税率,购退单价
4249,2025-10-10,11080833,梅毒螺旋体抗体检测试剂盒（）,条型单人份:50人份/盒,盒,体外诊断产品,体外诊断试剂,试剂,临床免疫检验专业,胶体金,...,浙江医疗,平台,ebs,试剂/耗材,定向采购,5.0,43.00,0.021500,0.13,NaN
4509,2025-10-11,11030068,反应杯,1000只,包,体外诊断产品,体外诊断试剂,配套试剂/耗材,临床免疫检验专业,发光,...,合肥迪赛金,华东片区,ebs,试剂/耗材,指定采购,40.0,231.81,0.927240,0.13,NaN
5097,2025-10-13,11080833,梅毒螺旋体抗体检测试剂盒（）,条型单人份:50人份/盒,盒,体外诊断产品,体外诊断试剂,试剂,临床免疫检验专业,胶体金,...,广迪科技,华南片区,ebs,试剂/耗材,定向采购,20.0,43.00,0.086000,0.13,NaN
5327,2025-10-13,11030068,反应杯,1000只,包,体外诊断产品,体外诊断试剂,配套试剂/耗材,临床免疫检验专业,发光,...,浙江医疗,平台,ebs,试剂/耗材,集中采购,8.0,237.96,0.190368,0.13,NaN
5650,2025-10-13,11081346,甲状旁腺素定标液 CalSet PTH,4×1.0mL(冻干品复溶体积),盒,体外诊断产品,体外诊断试剂,定标,临床免疫检验专业,发光,...,温州迪安,华东片区,nc,试剂/耗材,集中采购合同,1.0,721.29,0.072129,0.13,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
352370,2026-09-24,11030068,反应杯,1000只,包,体外诊断产品,体外诊断试剂,配套试剂/耗材,临床免疫检验专业,发光,...,上海迪浦康,华东片区,ebs,试剂/耗材,集中采购,8.0,237.96,0.190368,0.13,NaN
352665,2026-09-24,11030068,反应杯,1000只,包,体外诊断产品,体外诊断试剂,配套试剂/耗材,临床免疫检验专业,发光,...,上海迪浦康,华东片区,ebs,试剂/耗材,集中采购,1.0,237.96,0.023796,0.13,NaN
353168,2026-09-26,11030068,反应杯,1000只,包,体外诊断产品,体外诊断试剂,配套试剂/耗材,临床免疫检验专业,发光,...,浙江医疗,平台,ebs,试剂/耗材,指定采购,20.0,386.75,0.773500,0.13,NaN
353630,2026-09-28,11000575,清洗液,500mlx4瓶/盒,盒,体外诊断产品,体外诊断试剂,配套试剂/耗材,临床免疫检验专业,发光,...,浙江医疗,平台,ebs,试剂/耗材,指定采购,5.0,240.00,0.120000,0.13,NaN


### 公司编码与名称

同一公司编码对应多个名称。

In [10]:
条件 = df.groupby("公司编码", dropna=False)["公司名称"].transform("nunique").gt(1)
明细 = df.loc[条件, 字段]
print(f"涉及：{len(明细):,} 行")
明细

涉及：0 行


,出入库日期,物料编码,物料名称,规格,主单位,一级分类,二级分类,三级分类,四级分类,五级分类,...,公司简称,大区,数据来源系统,设备试剂分类,采购类型名称,数量,采购单价,采购金额(万),税率,购退单价


### 数量与采购金额

退货及有数量但金额为零的记录，保留。

In [11]:
明细 = df.loc[(df["数量"] < 0) | ((df["数量"] != 0) & (df["采购金额(万)"] == 0)), 字段]
print(f"涉及：{len(明细):,} 行")
明细

涉及：41,957 行


,出入库日期,物料编码,物料名称,规格,主单位,一级分类,二级分类,三级分类,四级分类,五级分类,...,公司简称,大区,数据来源系统,设备试剂分类,采购类型名称,数量,采购单价,采购金额(万),税率,购退单价
2,2025-10-01,11067963,生长激素测定试剂盒（）,100测试/盒,盒,体外诊断产品,体外诊断试剂,试剂,临床免疫检验专业,发光,...,上海迪安,华东片区,nc,试剂/耗材,定向采购合同,2.0,0.0,0.0000,0.13,NaN
11,2025-10-01,11068060,癌抗原15-3测定试剂盒（）,100人份/盒,盒,体外诊断产品,体外诊断试剂,试剂,临床免疫检验专业,发光,...,南京迪安,华东片区,nc,试剂/耗材,定向采购合同,4.0,0.0,0.0000,0.13,NaN
14,2025-10-01,11068206,免疫检验系统用底物液,2瓶/盒(底物液A:450mL/瓶×1瓶、底物液B:450mL/瓶×1瓶),盒,体外诊断产品,体外诊断试剂,配套试剂/耗材,临床免疫检验专业,发光,...,杭州迪安,华东片区,nc,试剂/耗材,定向采购合同,100.0,0.0,0.0000,0.13,NaN
18,2025-10-01,11059307,预激发液（适配机型 Wan200+）,200mL/瓶,瓶,体外诊断产品,体外诊断试剂,配套试剂/耗材,临床免疫检验专业,发光,...,杭州迪安,华东片区,nc,试剂/耗材,定向采购合同,100.0,0.0,0.0000,0.13,NaN
19,2025-10-01,11018915,激发液,900ml/瓶，4瓶/盒,箱,体外诊断产品,体外诊断试剂,配套试剂/耗材,临床免疫检验专业,发光,...,重庆迪安,西南片区,nc,试剂/耗材,集中采购合同,1.0,0.0,0.0000,0.13,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356098,2026-09-30,11020991,EB 病毒核心抗原 IgA 检测试剂盒（）,2×50人份/盒,盒,体外诊断产品,体外诊断试剂,试剂,临床免疫检验专业,发光,...,迪安诊断,平台,nc,试剂/耗材,集中采购合同,3.0,0.0,0.0000,0.13,NaN
356134,2026-09-30,11009829,样本稀释液,10L,桶,体外诊断产品,体外诊断试剂,配套试剂/耗材,临床血液与体液检验专业,粪便类,...,浙江医疗,平台,ebs,试剂/耗材,指定采购,-1.0,NaN,-0.1775,0.13,1775.0
356145,2026-09-30,11033701,清洗缓冲液（i 3000机型）,2L×4,箱,体外诊断产品,体外诊断试剂,配套试剂/耗材,临床免疫检验专业,发光,...,济南医疗,华中片区,ebs,试剂/耗材,定向采购,1.0,0.0,0.0000,0.13,NaN
356152,2026-09-30,11027570,巨细胞病毒 IgG 抗体非定值质控品,阳性：2×2mL，阴性：2×2mL,盒,体外诊断产品,体外诊断试剂,质控,临床免疫检验专业,发光,...,迪安诊断,平台,nc,试剂/耗材,集中采购合同,3.0,0.0,0.0000,0.13,NaN


### 已确认字段相同的记录

仅为重复候选；原始整行重复待业务确认，暂不剔除。

In [12]:
明细 = df.loc[df[字段].duplicated(keep=False), 字段]
print(f"重复候选：{len(明细):,} 行")
明细

重复候选：19,532 行


,出入库日期,物料编码,物料名称,规格,主单位,一级分类,二级分类,三级分类,四级分类,五级分类,...,公司简称,大区,数据来源系统,设备试剂分类,采购类型名称,数量,采购单价,采购金额(万),税率,购退单价
5,2025-10-01,11041618,乙型肝炎病毒表面抗体测定试剂盒,2×100测试/盒,盒,体外诊断产品,体外诊断试剂,试剂,临床免疫检验专业,发光,...,山西迪安,华北片区,nc,试剂/耗材,定向采购合同,5.0,278.00,0.139000,0.13,NaN
8,2025-10-01,11048409,甲胎蛋白测定试剂盒 AFP Reagent Kit（）,Alinity: 2×100 测试/盒,盒,体外诊断产品,体外诊断试剂,试剂,临床免疫检验专业,发光,...,杭州迪安,华东片区,nc,试剂/耗材,定向采购合同,2.0,1144.64,0.228928,0.13,NaN
74,2025-10-01,11026194,布鲁氏菌抗体检测试剂盒（）,100人份/盒(5ml×4瓶),盒,体外诊断产品,体外诊断试剂,试剂,临床微生物检验专业,微生物,...,山西迪安,华北片区,nc,试剂/耗材,定向采购合同,5.0,500.00,0.250000,0.13,NaN
84,2025-10-01,11048031,Brilliant Violet 421™anti-human CD19 Antibody,100 tests,盒,体外诊断产品,体外诊断试剂,试剂,临床血液与体液检验专业,流式细胞,...,北京迪安,华北片区,nc,试剂/耗材,定向采购合同,1.0,3149.00,0.314900,0.13,NaN
100,2025-10-01,11049959,营养琼脂培养基,90mm 10块/包,包,体外诊断产品,体外诊断试剂,试剂,临床微生物检验专业,微生物,...,成都迪安,西南片区,nc,试剂/耗材,定向采购合同,128.0,15.00,0.192000,0.13,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356051,2026-09-30,11004859,血细胞分析仪用质控物（6800）,"BC-6D,中值:4.5mL×1",支,体外诊断产品,体外诊断试剂,质控,临床血液与体液检验专业,血常规,...,迪安诊断,平台,nc,试剂/耗材,集中采购合同,4.0,258.00,0.103200,0.13,NaN
356093,2026-09-30,12021611,实时荧光定量 PCR 分析仪,SLAN-48S,台,体外诊断产品,体外诊断设备,检测设备,分子遗传检验设备,核酸扩增仪（定量）,...,广迪科技,华南片区,ebs,设备/配件,NaN,1.0,0.00,0.000000,0.13,NaN
356110,2026-09-30,11042657,雌二醇测定试剂盒 Estradiol Reagent Kit（适配 i2000）,ARCHITECT:1x100测试/盒,盒,体外诊断产品,体外诊断试剂,试剂,临床免疫检验专业,发光,...,基因工程B,平台,ebs,试剂/耗材,指定采购,1.0,629.00,0.062900,0.13,NaN
356121,2026-09-30,11044538,睾酮测定试剂盒 2nd Generation Testosterone Reagent Ki...,ARCHITECT:1×100测试/盒,盒,体外诊断产品,体外诊断试剂,试剂,临床免疫检验专业,发光,...,基因工程B,平台,ebs,试剂/耗材,指定采购,1.0,629.00,0.062900,0.13,NaN


## 4. 已确认清洗逻辑

- 文本去首尾空格及控制字符。
- 财务供应商类型空值填“外部单位”，更名为“供应商类型”。
- 采购单价空值用购退单价补充。
- 采购类型名称归一为“采购模式”，空值标“未标注”。
- 出入库日期转为日期，重复行保留。

以上规则已在 `clean_inbound` 中实现；负单价修正规则尚未实现。

In [13]:
from clean.procurement_inbound import clean_inbound

cleaned, 台账 = clean_inbound(df)
print(f"行数：{len(df):,} → {len(cleaned):,}；含税金额差：{cleaned['采购金额(万)'].sum() - df['采购金额(万)'].sum():,.6f} 万元")

行数：356,164 → 356,164；含税金额差：0.000000 万元


## 5. 清洗后完整数据

In [14]:
cleaned

,出入库日期,物料编码,物料名称,规格,主单位,一级分类,二级分类,三级分类,四级分类,五级分类,...,公司简称,大区,数据来源系统,设备试剂分类,采购模式,数量,采购单价,采购金额(万),税率,updated_at
0,2025-10-01,11030092,一次性使用无菌注射器(带针),"2ml,(200支/盒)",支,通用耗材,通用耗材,通用耗材,无分类,无分类,...,武汉迪安,华中片区,nc,试剂/耗材,自行采购,200.0,0.24,0.00480,0.13,2026-10-30T08:57:34+08:00
1,2025-10-01,11059447,铁测定试剂盒 L-Type Fe N（）,LABOSPECT专用包装:4×(60mL缓冲液(试剂1)、21mL显色液(试剂2))[4×...,盒,体外诊断产品,体外诊断试剂,试剂,临床化学检验专业,生化,...,南京迪安,华东片区,nc,试剂/耗材,定向采购,1.0,720.00,0.07200,0.13,2026-10-30T08:57:34+08:00
2,2025-10-01,11067963,生长激素测定试剂盒（）,100测试/盒,盒,体外诊断产品,体外诊断试剂,试剂,临床免疫检验专业,发光,...,上海迪安,华东片区,nc,试剂/耗材,定向采购,2.0,0.00,0.00000,0.13,2026-10-30T08:57:34+08:00
3,2025-10-01,11059338,总前列腺特异性抗原测定试剂盒（适配机型 Wan200+）,200人份/盒(含校准品),盒,体外诊断产品,体外诊断试剂,试剂,临床免疫检验专业,发光,...,杭州迪安,华东片区,nc,试剂/耗材,定向采购,1.0,680.00,0.06800,0.13,2026-10-30T08:57:34+08:00
4,2025-10-01,11020942,游离前列腺特异性抗原测定试剂盒（）,2×50人份/盒,盒,体外诊断产品,体外诊断试剂,试剂,临床免疫检验专业,发光,...,迪安诊断,平台,nc,试剂/耗材,集中采购,10.0,280.00,0.28000,0.13,2026-10-30T08:57:34+08:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356159,2026-09-30,11031673,CD33抗体试剂（FITC）,50检测人份,盒,体外诊断产品,体外诊断试剂,试剂,临床血液与体液检验专业,流式细胞,...,广迪科技,华南片区,ebs,试剂/耗材,定向采购,2.0,1027.00,0.20540,0.13,2026-10-30T08:57:34+08:00
356160,2026-09-30,11083052,人乳头瘤病毒核酸分型检测试剂盒,48人份/盒,盒,体外诊断产品,体外诊断试剂,试剂,临床细胞分子遗传学专业,PCR,...,云南迪安,西南片区,nc,试剂/耗材,指定采购,3.0,9547.20,2.86416,0.13,2026-10-30T08:57:34+08:00
356161,2026-09-30,11085635,抗β2糖蛋白 I 抗体测定试剂盒（二代）,2×50人份/盒,盒,体外诊断产品,体外诊断试剂,试剂,临床免疫检验专业,发光,...,迪安诊断,平台,nc,试剂/耗材,集中采购,5.0,552.50,0.27625,0.13,2026-10-30T08:57:34+08:00
356162,2026-09-30,11086260,胃泌素17测定试剂盒,100人份/盒(含校准品),盒,体外诊断产品,体外诊断试剂,试剂,临床免疫检验专业,发光,...,基因工程B,平台,ebs,试剂/耗材,指定采购,4.0,2176.00,0.87040,0.13,2026-10-30T08:57:34+08:00
